# Baseline Grid-Based Pothole Detector

This notebook demonstrates training a baseline convolutional neural network for pothole detection using a grid-based approach. The model divides images into an 8×8 grid and predicts pothole presence in each grid cell.

## 1. Import Required Libraries

In [ ]:
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from pathlib import Path
import kagglehub
import xml.etree.ElementTree as ET
import glob
import os
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Define PatchPotholeDataset Class

The dataset class loads images, resizes them to 256×256, and creates 8×8 grid targets with bounding box annotations.

In [ ]:
class PatchPotholeDataset(torch.utils.data.Dataset):
    """Grid-based pothole dataset for patch-level detection.
    
    Divides images into an NxN grid and creates binary targets for each grid cell,
    marking cells that contain pothole bounding boxes.
    
    Args:
        df: DataFrame with columns ['file', 'xmin', 'ymin', 'xmax', 'ymax']
        img_size: Target image size (default: 256)
        grid_size: Grid dimension (default: 8 for 8x8 grid)
    """
    
    def __init__(self, df, img_size=256, grid_size=8):
        self.df = df
        self.images = df['file'].unique()
        self.img_size = img_size
        self.grid_size = grid_size
        self.cell_size = img_size / grid_size

    def __getitem__(self, idx):
        img_path = self.images[idx]
        
        # Load and convert image
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        orig_h, orig_w = img.shape[:2]
        
        # Resize image to fixed dimensions
        img_resized = cv2.resize(img, (self.img_size, self.img_size))
        
        # Normalize and convert to PyTorch tensor (C, H, W)
        img_tensor = torch.from_numpy(img_resized).permute(2, 0, 1).float() / 255.0

        # Create empty grid target (1, grid_size, grid_size) filled with zeros
        target_grid = torch.zeros((1, self.grid_size, self.grid_size), dtype=torch.float32)
        
        # Get all annotations for this image
        img_data = self.df[self.df["file"] == img_path]
        
        # Mark grid cells that contain bounding boxes
        for _, row in img_data.iterrows():
            # Scale bounding box coordinates to resized image
            xmin = int(row['xmin'] * (self.img_size / orig_w))
            ymin = int(row['ymin'] * (self.img_size / orig_h))
            xmax = int(row['xmax'] * (self.img_size / orig_w))
            ymax = int(row['ymax'] * (self.img_size / orig_h))
            
            # Calculate which grid cells intersect with the bounding box
            grid_xmin = max(0, int(xmin // self.cell_size))
            grid_ymin = max(0, int(ymin // self.cell_size))
            grid_xmax = min(self.grid_size - 1, int(xmax // self.cell_size))
            grid_ymax = min(self.grid_size - 1, int(ymax // self.cell_size))
            
            # Set 1 in corresponding grid cells
            target_grid[0, grid_ymin:grid_ymax+1, grid_xmin:grid_xmax+1] = 1.0

        return img_tensor, target_grid

    def __len__(self):
        return len(self.images)

print("PatchPotholeDataset class defined successfully!")

## 3. Define PotholePatchNet Model Architecture

The model consists of:
- **Feature extraction**: 5 convolutional blocks with max pooling (256×256 → 8×8)
- **Classification head**: 1×1 convolutions for per-cell binary predictions

In [ ]:
class PotholePatchNet(nn.Module):
    """Baseline CNN for grid-based pothole detection.
    
    This architecture detects potholes by dividing the image into an 8x8 grid
    and predicting presence/absence of potholes in each grid cell.
    
    Input:  (B, 3, 256, 256) - RGB images
    Output: (B, 1, 8, 8) - Binary predictions per grid cell
    """
    
    def __init__(self):
        super(PotholePatchNet, self).__init__()
        
        # Feature extraction: progressively downsample from 256x256 to 8x8
        self.features = nn.Sequential(
            # Input: 3 x 256 x 256
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # -> 128x128
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # -> 64x64
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # -> 32x32
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # -> 16x16
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # -> 8x8
        )
        
        # Classification head: 1x1 convolutions for per-cell predictions
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 1, kernel_size=1)  # Binary output per cell
        )

    def forward(self, x):
        """
        Args:
            x: Input tensor of shape (B, 3, 256, 256)
        
        Returns:
            Output tensor of shape (B, 1, 8, 8) with logits for each grid cell
        """
        x = self.features(x)
        x = self.classifier(x)
        return x

print("PotholePatchNet model defined successfully!")

## 4. Load Data and Initialize Training Components

Parse annotations from XML files, create dataset and dataloaders, and initialize model and optimizer.

In [ ]:
# Define helper function to parse XML annotations
LABEL_MAP = {
    "minor_pothole": 1,
    "medium_pothole": 2,
    "major_pothole": 3
}

def parse_xmls(path):
    """Parse pothole annotations from XML files."""
    df = []
    
    img_dir = os.path.join(path, 'images')
    annots_path = os.path.join(path, 'annotations/*.xml')
    
    for xml_file in glob.glob(annots_path):
        tree = ET.parse(xml_file)
        root = tree.getroot()
        
        filename = root.find("filename").text
        img_path = os.path.join(img_dir, filename)
        
        for obj in root.findall("object"):
            severity = obj.find('name').text.lower()
            
            df.append({
                "file": img_path,
                "xmin": int(obj.find('bndbox/xmin').text),
                "ymin": int(obj.find('bndbox/ymin').text),
                "xmax": int(obj.find('bndbox/xmax').text),
                "ymax": int(obj.find('bndbox/ymax').text),
                "label": LABEL_MAP.get(severity, 1)
            })
    
    return pd.DataFrame(df)

# Load dataset
print("Downloading and parsing pothole dataset...")
data_path = kagglehub.dataset_download("idanbaru/annotated-potholes-with-severity-levels")
df = parse_xmls(data_path)
print(f"✓ Loaded {len(df)} annotations from {len(df['file'].unique())} images")

# Create dataset
print("\nCreating PatchPotholeDataset...")
dataset = PatchPotholeDataset(df, img_size=256, grid_size=8)
print(f"✓ Dataset created with {len(dataset)} images")

# Split dataset: 80% train, 10% val, 10% test
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size])
print(f"✓ Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

# Create dataloaders
BATCH_SIZE = 16
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"\n✓ DataLoaders created (batch_size={BATCH_SIZE})")

# Initialize model, loss, and optimizer
model = PotholePatchNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"\n✓ Model initialized on {device}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Train the Model

Execute the training loop with loss tracking and early validation.

In [ ]:
# Training configuration
EPOCHS = 10
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

print("Starting training...\n")
print(f"{'Epoch':<8} {'Train Loss':<12} {'Val Loss':<12} {'Val Acc':<10}")
print("-" * 45)

for epoch in range(EPOCHS):
    # Training phase
    model.train()
    train_loss = 0.0
    
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss = train_loss / len(train_loader)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, targets)
            val_loss += loss.item()
            
            # Calculate accuracy
            predictions = (torch.sigmoid(outputs) > 0.5).float()
            val_correct += (predictions == targets).sum().item()
            val_total += targets.numel()
    
    val_loss = val_loss / len(val_loader)
    val_acc = val_correct / val_total
    
    # Store history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"[{epoch+1:<7}] {train_loss:<11.4f} {val_loss:<11.4f} {val_acc:<9.4f}")

print("-" * 45)
print("Training completed!\n")

## 6. Visualize Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history['val_acc'], label='Val Accuracy', marker='o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best validation accuracy: {max(history['val_acc']):.4f}")
print(f"Final validation loss: {history['val_loss'][-1]:.4f}")

## 7. Evaluate Model on Test Set

In [ ]:
# Evaluate on test set
model.eval()
test_loss = 0.0
test_correct = 0
test_total = 0

print("Evaluating on test set...")
with torch.no_grad():
    for images, targets in test_loader:
        images, targets = images.to(device), targets.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, targets)
        test_loss += loss.item()
        
        # Calculate metrics
        predictions = (torch.sigmoid(outputs) > 0.5).float()
        test_correct += (predictions == targets).sum().item()
        test_total += targets.numel()
        
        # Calculate per-cell precision, recall, F1
        tp = ((predictions == 1) & (targets == 1)).sum().item()
        fp = ((predictions == 1) & (targets == 0)).sum().item()
        fn = ((predictions == 0) & (targets == 1)).sum().item()

test_loss = test_loss / len(test_loader)
test_acc = test_correct / test_total

print(f"\n{'Test Metrics':^40}")
print("-" * 40)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print("-" * 40)

## 8. Visualize Predictions on Sample Images

In [ ]:
def visualize_predictions(model, dataset, num_samples=3, grid_size=8, img_size=256):
    """Visualize model predictions on sample images."""
    
    model.eval()
    cell_size = img_size / grid_size
    
    # Sample random images
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for row, idx in enumerate(indices):
        img_tensor, target_grid = dataset[idx]
        
        # Move to device and add batch dimension
        img_batch = img_tensor.unsqueeze(0).to(device)
        
        # Get prediction
        with torch.no_grad():
            output = model(img_batch)
            pred_grid = torch.sigmoid(output).squeeze().cpu().numpy()
        
        # Convert image to numpy for visualization
        img_np = img_tensor.permute(1, 2, 0).numpy()
        target_np = target_grid.squeeze().cpu().numpy()
        
        # Plot original image
        axes[row, 0].imshow(img_np)
        axes[row, 0].set_title('Original Image')
        axes[row, 0].grid(True, alpha=0.3)
        
        # Plot target grid
        im1 = axes[row, 1].imshow(target_np, cmap='RdYlGn', vmin=0, vmax=1)
        axes[row, 1].set_title('Ground Truth Grid')
        axes[row, 1].grid(True, alpha=0.3)
        plt.colorbar(im1, ax=axes[row, 1])
        
        # Plot prediction grid
        im2 = axes[row, 2].imshow(pred_grid, cmap='RdYlGn', vmin=0, vmax=1)
        axes[row, 2].set_title('Predicted Grid')
        axes[row, 2].grid(True, alpha=0.3)
        plt.colorbar(im2, ax=axes[row, 2])
    
    plt.tight_layout()
    plt.show()

print("Visualizing predictions on sample images...\n")
visualize_predictions(model, test_ds, num_samples=3)

## 9. Save and Load Model Checkpoint

In [ ]:
# Save model checkpoint
checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)
checkpoint_path = checkpoint_dir / "pothole_patch_net_baseline.pt"

torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': EPOCHS,
    'history': history,
    'hyperparameters': {
        'img_size': 256,
        'grid_size': 8,
        'batch_size': BATCH_SIZE,
        'learning_rate': 0.001,
    }
}, checkpoint_path)

print(f"✓ Model checkpoint saved to {checkpoint_path}")

# Load model checkpoint
checkpoint = torch.load(checkpoint_path)
model_loaded = PotholePatchNet().to(device)
model_loaded.load_state_dict(checkpoint['model_state_dict'])

print(f"✓ Model checkpoint loaded successfully")
print(f"  Epochs trained: {checkpoint['epoch']}")
print(f"  Hyperparameters: {checkpoint['hyperparameters']}")

## 10. Summary and Next Steps

### Model Architecture Summary
- **Input**: 256×256 RGB images
- **Feature Extraction**: 5 convolutional blocks with max pooling (progressively reduces to 8×8)
- **Classification**: 1×1 convolutions for binary pothole predictions per grid cell
- **Total Parameters**: ~600K
- **Output**: 8×8 grid of binary predictions

### Results
- **Test Accuracy**: {:.4f}
- **Test Loss**: {:.4f}

### Next Steps for Improvement
1. **Data Augmentation**: Add rotation, flipping, brightness adjustments
2. **Architecture Enhancements**: Try deeper networks or residual connections
3. **Loss Function Tuning**: Experiment with focal loss or weighted BCE
4. **Hyperparameter Search**: Optimize learning rate, batch size, grid size
5. **Model Ensembling**: Combine multiple models for better predictions
6. **Post-processing**: Apply CRF or morphological operations to smooth predictions
7. **Integration with Lightning**: Refactor to use PyTorch Lightning for scalability
8. **Evaluation Metrics**: Implement precision, recall, F1-score per severity level